# Mouse PBMC (Han et al. 2018 MCA, K=9) -- combined comparison

Reproduces Table 1 of the paper (Section 6): 6 `PeripheralBlood_<i>`
batches (each a turn as target, leave-one-batch-out), K=9 cell types,
batch sizes ranging 135-3201 with several cell types entirely absent
from some batches -- see `common.py`'s docstring for why this dataset
was included.

`run_mouse_pbmc_comparison.py` doesn't do a one-source-at-a-time
breakdown: every method (besides `target_only`) always pools all 5 other
batches as its source, so there's no `source` column here to filter on --
one row per `(target, method)`.

7 methods (`common.METHOD_LABELS`), matching Table 1's columns exactly:
target-only, multi-source pooled, and adaptive multi-source (Algorithm 3);
the pooled estimator **`target_source_pooled_capped`**
(`target_source_pooled_subspace_estimate(..., restrict_basis_rank=True)`,
matching Algorithm 4 -- the pooled projection subspace SVD-truncated to
its leading K=9 directions, labeled plain "Pooled"); and the three
comparator methods, TL-GMM, NMF (`scrna`), and GDEC.

Reads `results/raw/*.csv`, reproduces the combined ARI/V-measure/misclustering
LaTeX table, a standalone ARI-only LaTeX table, and misclustering/ARI bar
charts for all 7 methods.

Standalone and read-only with respect to `results/raw/` -- rerun any time
after adding/replacing files there. Outputs land in `results/combined/`:
`mouse_pbmc_comparison_results.csv`, `mouse_pbmc_combined_table.tex`,
`mouse_pbmc_ari_table.tex`, `mouse_pbmc_misclustering.pdf`, `mouse_pbmc_ari.pdf`.

In [ ]:
import glob, os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import common


## Load, combine, and check completeness

Reads every `results/raw/*.csv` and checks completeness against the
expected 42-row grid (6 batches x 7 methods, see
`run_mouse_pbmc_comparison.py`'s `task_grid()`) before doing anything else.


In [ ]:
RESULTS_DIR = "results"
RAW_DIR = os.path.join(RESULTS_DIR, "raw")
OUT_DIR = os.path.join(RESULTS_DIR, "combined")
os.makedirs(OUT_DIR, exist_ok=True)

raw_paths = sorted(glob.glob(os.path.join(RAW_DIR, "*.csv")))
print(f"Found {len(raw_paths)} raw result files in {RAW_DIR}")
assert raw_paths, f"No CSVs found in {RAW_DIR}"

INCLUDED_METHODS = list(common.METHOD_LABELS.keys())  # all 7
LABELS = common.METHOD_LABELS


def expected_grid():
    """Mirrors run_mouse_pbmc_comparison.py's task_grid(): every (target, method) pair."""
    return {(target, method) for target in common.BATCHES for method in INCLUDED_METHODS}


expected = expected_grid()
found = set()
rows = []
for path in raw_paths:
    row = pd.read_csv(path).iloc[0]
    rows.append(row.to_dict())
    found.add((row["target"], row["method"]))

missing = expected - found
if missing:
    print(f"WARNING: {len(missing)}/{len(expected)} (target, method) results missing from {RAW_DIR}:")
    for target, method in sorted(missing):
        print(f"  {target}, {method}")
else:
    print(f"All {len(expected)} (target, method) combinations present")

# No `source` column in this driver's output (see intro markdown) -- one row
# per (target, method) already, so this is directly the "all-sources" table
# the later cells pivot on (named results_all to match the other notebooks'
# convention).
results_all = pd.DataFrame(rows)
combined_path = os.path.join(OUT_DIR, "mouse_pbmc_comparison_results.csv")
results_all.to_csv(combined_path, index=False)
print(f"Wrote {combined_path}")
results_all


## Summary tables (method x target batch)


In [ ]:
for metric in ["misclustering", "ari", "v_measure"]:
    print(f"\n--- {metric} ---")
    display(results_all.pivot(index="method", columns="target", values=metric)
                        .reindex(index=INCLUDED_METHODS, columns=common.BATCHES))


## Combined LaTeX booktabs table

One column group per target batch, each split into three subcolumns -- ARI,
V-measure, $\mathcal{L}_{\mathrm{mult}}$ (misclustering error) -- one row
per method, 7 methods total. Best value per subcolumn bolded. 1 + 6*3 = 19
columns, wrapped in `\resizebox`.


In [ ]:
SUBCOLS = [("ari", "ARI", "high"), ("v_measure", "V-measure", "high"),
           ("misclustering", "$\\mathcal{L}_{\\mathrm{mult}}$", "low")]


def make_combined_latex_table(results: pd.DataFrame, methods, labels, caption: str = "", label: str = "") -> str:
    """Single booktabs table: rows = methods, columns = (batch, metric) for
    every batch x {ari, v_measure, misclustering} pair, best value per
    (batch, metric) subcolumn bolded (ari/v_measure: higher is better;
    misclustering: lower is better). `results` should already be filtered to
    one row per (method, target) -- e.g. `results_all` -- since `pivot`
    errors on duplicate (method, target) pairs."""
    pivots = {
        metric: results.pivot(index="method", columns="target", values=metric)
                        .reindex(index=methods, columns=common.BATCHES)
        for metric, _, _ in SUBCOLS
    }

    n_sub = len(SUBCOLS)
    lines = []
    lines.append("\\begin{table}[t]")
    lines.append("\\centering")
    lines.append("\\resizebox{\\textwidth}{!}{%")
    lines.append("\\begin{tabular}{l" + "ccc" * len(common.BATCHES) + "}")
    lines.append("\\toprule")

    header1 = [""]
    cmidrules = []
    col = 2
    for batch in common.BATCHES:
        header1.append(f"\\multicolumn{{{n_sub}}}{{c}}{{{batch.replace('_', chr(92) + '_')}}}")
        cmidrules.append(f"\\cmidrule(lr){{{col}-{col + n_sub - 1}}}")
        col += n_sub
    lines.append(" & ".join(header1) + " \\\\")
    lines.append(" ".join(cmidrules))

    header2 = ["Method"] + [sub_label for _ in common.BATCHES for _, sub_label, _ in SUBCOLS]
    lines.append(" & ".join(header2) + " \\\\")
    lines.append("\\midrule")

    for method in methods:
        cells_ = []
        for batch in common.BATCHES:
            for metric, _, direction in SUBCOLS:
                pivot = pivots[metric]
                val = pivot.loc[method, batch]
                best = pivot[batch].min() if direction == "low" else pivot[batch].max()
                cell = f"{val:.3f}"
                if np.isclose(val, best):
                    cell = f"\\textbf{{{cell}}}"
                cells_.append(cell)
        label_str = labels[method].replace("_", "\\_").replace("+", "$+$")
        lines.append(f"{label_str} & " + " & ".join(cells_) + " \\\\")

    lines.append("\\bottomrule")
    lines.append("\\end{tabular}%")
    lines.append("}")
    if caption:
        lines.append(f"\\caption{{{caption}}}")
    if label:
        lines.append(f"\\label{{{label}}}")
    lines.append("\\end{table}")
    return "\n".join(lines)


caption = ("ARI, V-measure, and misclustering error ($\\mathcal{L}_{\\mathrm{mult}}$) on the mouse "
           "PBMC atlas (Han et al. 2018 MCA, K=9), leave-one-batch-out: all 7 methods, with ``Pooled'' "
           "the rank-capped (K=9) target+source pooled-subspace estimator. ARI/V-measure: higher is "
           "better; $\\mathcal{L}_{\\mathrm{mult}}$: lower is better. Best value per subcolumn in bold.")
combined_tex = make_combined_latex_table(results_all, INCLUDED_METHODS, LABELS, caption=caption,
                                          label="tab:mouse_pbmc_combined")
print(combined_tex)

table_path = os.path.join(OUT_DIR, "mouse_pbmc_combined_table.tex")
with open(table_path, "w") as f:
    f.write(combined_tex)
    f.write("\n")
print(f"Wrote {table_path}")


## Figures: misclustering error and ARI by target batch

Grouped bar charts, 7 methods, 6 batch groups, using a fixed categorical
color per method (consistent across both figures).

In [ ]:
METHOD_COLORS = {
    "target_only": "#2a78d6",                   # slot 1 blue
    "target_source_pooled_capped": "#eb6834",   # slot 2 orange ("Pooled")
    "multi_source_pooled": "#1baf7a",           # slot 3 aqua
    "adaptive_multi_source": "#eda100",         # slot 4 yellow
    "tlgmm": "#008300",                         # slot 6 green
    "scrna": "#4a3aa7",                         # slot 7 violet
    "gdec_gcnfree": "#e34948",                  # slot 8 red
}
# Same left-to-right order the colors above were validated in.
METHOD_ORDER = list(METHOD_COLORS.keys())


def plot_metric_bar_chart(metric: str, ylabel: str, title_suffix: str, out_name: str):
    pivot = results_all.pivot(index="method", columns="target", values=metric).reindex(
        index=METHOD_ORDER, columns=common.BATCHES
    )

    n_methods = len(METHOD_ORDER)
    x = np.arange(len(common.BATCHES))
    width = 0.8 / n_methods

    fig, ax = plt.subplots(figsize=(13, 5.5))
    for i, method in enumerate(METHOD_ORDER):
        offset = (i - (n_methods - 1) / 2) * width
        vals = pivot.loc[method].values
        bars = ax.bar(x + offset, vals, width, label=LABELS[method], color=METHOD_COLORS[method])
        ax.bar_label(bars, fmt="%.2f", padding=2, fontsize=6, rotation=90)

    ax.set_xticks(x)
    ax.set_xticklabels(common.BATCHES, rotation=15)
    ax.set_ylabel(ylabel)
    ax.set_title(f"Mouse PBMC (K=9): {title_suffix} by target batch")
    ymin = min(0.0, pivot.values.min() * 1.1)
    ax.set_ylim(ymin, max(pivot.values.max() * 1.25, 0.05))
    ax.spines[["top", "right"]].set_visible(False)
    ax.grid(True, axis="y", alpha=0.25)
    ax.legend(frameon=False, fontsize=8, ncol=4, loc="upper center", bbox_to_anchor=(0.5, -0.16))
    fig.tight_layout()

    fig_path = os.path.join(OUT_DIR, out_name)
    fig.savefig(fig_path, bbox_inches="tight")
    print(f"Wrote {fig_path}")
    plt.show()


plot_metric_bar_chart("misclustering", "misclustering error (lower is better)",
                       "misclustering error", "mouse_pbmc_misclustering.pdf")
plot_metric_bar_chart("ari", "Adjusted Rand Index (higher is better)",
                       "ARI", "mouse_pbmc_ari.pdf")


## Standalone ARI-only LaTeX table

One column per target batch, one row per method -- just the ARI subcolumn
of the combined table above, pulled out on its own (methods x batches,
best value per batch bolded).

In [ ]:
def make_single_metric_latex_table(results: pd.DataFrame, methods, labels, metric: str,
                                    direction: str, caption: str = "", label: str = "") -> str:
    """Single booktabs table: rows = methods, columns = target batches, one
    value per (method, batch) for `metric` only. Best value per column
    bolded (direction: 'high' or 'low'). `results` should already be
    filtered to one row per (method, target) -- e.g. `results_all`."""
    pivot = results.pivot(index="method", columns="target", values=metric).reindex(
        index=methods, columns=common.BATCHES
    )

    lines = []
    lines.append("\\begin{table}[t]")
    lines.append("\\centering")
    lines.append("\\begin{tabular}{l" + "c" * len(common.BATCHES) + "}")
    lines.append("\\toprule")
    header = ["Method"] + [batch.replace("_", "\\_") for batch in common.BATCHES]
    lines.append(" & ".join(header) + " \\\\")
    lines.append("\\midrule")

    for method in methods:
        cells_ = []
        for batch in common.BATCHES:
            val = pivot.loc[method, batch]
            best = pivot[batch].min() if direction == "low" else pivot[batch].max()
            cell = f"{val:.3f}"
            if np.isclose(val, best):
                cell = f"\\textbf{{{cell}}}"
            cells_.append(cell)
        label_str = labels[method].replace("_", "\\_").replace("+", "$+$")
        lines.append(f"{label_str} & " + " & ".join(cells_) + " \\\\")

    lines.append("\\bottomrule")
    lines.append("\\end{tabular}")
    if caption:
        lines.append(f"\\caption{{{caption}}}")
    if label:
        lines.append(f"\\label{{{label}}}")
    lines.append("\\end{table}")
    return "\n".join(lines)


ari_caption = ("Adjusted Rand Index (ARI) on the mouse PBMC atlas (Han et al. 2018 MCA, K=9), "
               "leave-one-batch-out: all 7 methods, with ``Pooled'' the rank-capped (K=9) "
               "target+source pooled-subspace estimator. Higher is better; best value per "
               "batch in bold.")
ari_tex = make_single_metric_latex_table(results_all, INCLUDED_METHODS, LABELS, metric="ari",
                                          direction="high", caption=ari_caption,
                                          label="tab:mouse_pbmc_ari")
print(ari_tex)

ari_table_path = os.path.join(OUT_DIR, "mouse_pbmc_ari_table.tex")
with open(ari_table_path, "w") as f:
    f.write(ari_tex)
    f.write("\n")
print(f"Wrote {ari_table_path}")